In [1]:
%%writefile predict.py

import pickle
from flask import Flask, request, jsonify

with open('models/lin_reg.bin', 'rb') as f:
    dv, model = pickle.load(f)

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    trip = request.get_json()
    
    features = {
        'PULocationID': str(trip['PULocationID']),
        'DOLocationID': str(trip['DOLocationID']),
        'trip_distance': trip['trip_distance']
    }
    
    X = dv.transform([features])
    duration = model.predict(X)[0]
    
    return jsonify({
        'duration_minutes': round(float(duration), 2)
    })

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=9696, debug=True)

Overwriting predict.py


In [2]:
%%writefile Dockerfile

FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir --timeout 100 -r requirements.txt

COPY predict.py .
COPY models/ models/

EXPOSE 9696

CMD ["python", "predict.py"]

Overwriting Dockerfile


In [3]:
%%writefile requirements.txt
flask
scikit-learn
numpy

Overwriting requirements.txt


In [4]:
!docker build -t nyc-taxi-duration:v1 .


[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.1s (1/3)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 249B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.12-slim        0.1s
 => [auth] library/python:pull token for registry-1.docker.io              0.0s
[+] Building 0.3s (2/3)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 249B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.12-slim        0.2s
 => [auth] library/python:pull token for registry-1.docker.io              0.0s
[+] Building 0.3s (3/3)                                          docker:default
 => [internal] load build definition fr

In [5]:
!docker run -d -p 9696:9696 --name nyc-taxi nyc-taxi-duration:v1
print("Container running on port 9696!")

1b5ec943dcb3eb4b5040ebeb047fb21db29e764f075036bc68ff13c7cb0b12e1
Container running on port 9696!


In [6]:
import requests

trip = {
    "PULocationID": 65,
    "DOLocationID": 233,
    "trip_distance": 6.20
}

response = requests.post("http://localhost:9696/predict", json=trip)
print(response.json())

{'duration_minutes': 23.9}
